In [ ]:
pip install datasets tensorflow scikit-learn pandas matplotlib

In [ ]:
import pandas as pd
import numpy as np

from datasets import load_dataset

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

from tensorflow.keras.models import Sequential

from tensorflow.keras.layers import Embedding
from tensorflow.keras.layers import SimpleRNN
from tensorflow.keras.layers import Dense

from sklearn.model_selection import train_test_split

from tensorflow.keras.utils import to_categorical

import matplotlib.pyplot as plt

In [ ]:
import pandas as pd

train_df = pd.read_csv(
    "train.csv",
    skiprows=1,
    header=None,
    names=['label','title','description']
)

test_df = pd.read_csv(
    "test.csv",
    skiprows=1,
    header=None,
    names=['label','title','description']
)

In [ ]:
train_df['text'] = train_df['title'] + " " + train_df['description']
test_df['text'] = test_df['title'] + " " + test_df['description']

In [ ]:
X_train = train_df['text']
X_test = test_df['text']

y_train = train_df['label']
y_test = test_df['label']

In [ ]:
y_train = y_train.astype(int) - 1
y_test = y_test.astype(int) - 1

In [ ]:
train_df['label'] = train_df['label'].astype(int) - 1
test_df['label'] = test_df['label'].astype(int) - 1

In [ ]:
from tensorflow.keras.preprocessing.text import Tokenizer

max_words = 20000

tokenizer = Tokenizer(num_words=max_words, oov_token="<OOV>")
tokenizer.fit_on_texts(X_train)

X_train_seq = tokenizer.texts_to_sequences(X_train)
X_test_seq = tokenizer.texts_to_sequences(X_test)

In [ ]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

max_len = 100

X_train_pad = pad_sequences(X_train_seq, maxlen=max_len, padding='post')
X_test_pad = pad_sequences(X_test_seq, maxlen=max_len, padding='post')

In [ ]:
from tensorflow.keras.utils import to_categorical

y_train = to_categorical(y_train, num_classes=4)
y_test = to_categorical(y_test, num_classes=4)

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, Dense

model = Sequential()

model.add(Embedding(20000, 128, input_length=max_len))

model.add(SimpleRNN(128, activation='tanh', return_sequences=True))
model.add(SimpleRNN(64))

model.add(Dense(64, activation='relu'))
model.add(Dense(4, activation='softmax'))

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [ ]:
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [ ]:
history = model.fit(
    X_train_pad,
    y_train,
    epochs=5,
    batch_size=128,
    validation_split=0.2
)

Epoch 1/5
750/750 ━━━━━━━━━━━━━━━━━━━━ 180s 233ms/step - accuracy: 0.3236 - loss: 1.3527 - val_accuracy: 0.4022 - val_loss: 1.3057
Epoch 2/5
750/750 ━━━━━━━━━━━━━━━━━━━━ 166s 222ms/step - accuracy: 0.4257 - loss: 1.2729 - val_accuracy: 0.4048 - val_loss: 1.3052
Epoch 3/5
750/750 ━━━━━━━━━━━━━━━━━━━━ 177s 236ms/step - accuracy: 0.3879 - loss: 1.3061 - val_accuracy: 0.4293 - val_loss: 1.2519
Epoch 4/5
750/750 ━━━━━━━━━━━━━━━━━━━━ 191s 221ms/step - accuracy: 0.5728 - loss: 1.0635 - val_accuracy: 0.5901 - val_loss: 1.0470
Epoch 5/5
750/750 ━━━━━━━━━━━━━━━━━━━━ 175s 234ms/step - accuracy: 0.6651 - loss: 0.8832 - val_accuracy: 0.6412 - val_loss: 0.9268


In [ ]:
loss, acc = model.evaluate(X_test_pad, y_test)
print("Test Accuracy:", acc)

238/238 ━━━━━━━━━━━━━━━━━━━━ 5s 20ms/step - accuracy: 0.6579 - loss: 0.8897
Test Accuracy: 0.6578947305679321


In [ ]:
import numpy as np

classes = ["World", "Sports", "Business", "Sci/Tech"]

news = ["India wins cricket world cup in a thrilling final"]

seq = tokenizer.texts_to_sequences(news)
pad = pad_sequences(seq, maxlen=max_len, padding='post')

pred = model.predict(pad)

print("Predicted Category:", classes[np.argmax(pred)])

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 496ms/step
Predicted Category: World


In [ ]:
import numpy as np

classes = ["World", "Sports", "Business", "Sci/Tech"]

news = ["Stock market crashes due to global economic slowdown"]

seq = tokenizer.texts_to_sequences(news)
pad = pad_sequences(seq, maxlen=max_len, padding='post')

pred = model.predict(pad)

print("Predicted Category:", classes[np.argmax(pred)])

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 76ms/step
Predicted Category: Business
